In [1]:
import sys
sys.path.append('../../')

from file_management import get_files_dir,check_save_file
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

import pandas as pd
from Bio import Entrez

Entrez.email = 'elisa.m.zavala@ntnu.no'

In [2]:
INPUT_DIR

'/Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Input'

In [3]:
# Load the ChEBI database into a Pandas DataFrame.

full_compounds_file = INPUT_DIR+'/Products/compounds.tsv'
full_compounds = pd.read_table(full_compounds_file, dtype=str, index_col='ID')
full_compounds = full_compounds.loc[:,'NAME']
full_compounds.head()

ID
9349        sulfonyldimethane
9352                 sulindac
9355               sulfuretin
9380                 syringin
9427    2-methylanthraquinone
Name: NAME, dtype: object

In [4]:
full_compounds.shape

(189584,)

In [5]:
# Load the ChEBI database into a Pandas DataFrame.

full_syn_file = INPUT_DIR+'/Products/names.tsv'
full_syn = pd.read_table(full_syn_file, dtype=str, index_col='COMPOUND_ID')
full_syn = full_syn.loc[full_syn.LANGUAGE=='en',['TYPE','NAME']]
full_syn.head()

,TYPE,NAME
COMPOUND_ID,,
16478,SYNONYM,"N-Acetyl-beta-D-glucosaminyl-1,6-(N-acetyl-bet..."
15947,SYNONYM,N-Acetyl-beta-D-glucosaminylamine
7853,SYNONYM,Oxyacanthine
15379,SYNONYM,Oxygen
15379,SYNONYM,O2


In [42]:
full_syn = pd.read_table(full_syn_file, dtype=str, index_col='COMPOUND_ID')


In [6]:
full_syn.shape

(337398, 2)

In [7]:
duplicates = full_syn.index[full_syn.index.duplicated()]
print(full_syn.loc[duplicates])


                   TYPE                                       NAME
COMPOUND_ID                                                       
15379           SYNONYM                                     Oxygen
15379           SYNONYM                                         O2
15379        IUPAC NAME                                   dioxygen
15379           SYNONYM                           molecular oxygen
15379           SYNONYM                                         O2
...                 ...                                        ...
60330           SYNONYM                                        ADO
60330           SYNONYM  2-methyl-2-(methylsulfanyl)propanaldoxime
60330              NAME  2-methyl-2-(methylsulfanyl)propanal oxime
60197           SYNONYM                                     BChl c
60197              NAME                    a bacteriochlorophyll c

[1977078 rows x 2 columns]


In [8]:
full_syn = full_syn.drop_duplicates()
full_syn = full_syn.groupby('COMPOUND_ID').apply(lambda x: x['NAME'].tolist())
full_syn.name = 'Synonym'

In [9]:
ChEBI = pd.concat([full_compounds,full_syn], axis=1)

In [10]:
ChEBI = ChEBI.dropna(how='all')

In [11]:
ChEBI.loc[17234].Synonym

['Glucose', 'glucose', 'Glukose', 'Glc', 'gluco-hexose', 'DL-glucose']

In [12]:
ChEBI.shape

(170158, 2)

In [18]:
ChEBI.dropna(subset=['Synonym']).to_json('rm_chebi.json')

In [40]:
lactic = ChEBI.NAME[ChEBI.NAME.str.contains('lactic')]

In [41]:
lactic

422                                         (S)-lactic acid
16003                    (R)-3-(4-hydroxyphenyl)lactic acid
16122             3-(4-hydroxy-3,5-diiodophenyl)lactic acid
16373                      (S)-3-(imidazol-5-yl)lactic acid
16444                                   2-acetyllactic acid
16712                                (S)-3-sulfolactic acid
17385                        3-(4-hydroxyphenyl)lactic acid
17807                    3-(3,4-dihydroxyphenyl)lactic acid
17943                              (S)-2-O-sulfolactic acid
23789                             dihydroxyphenylactic acid
24735                              hydroxyphenyllactic acid
24813                             3-(indol-3-yl)lactic acid
24827                                     indolelactic acid
24999                                          lactic acids
25190                                   mercaptolactic acid
25396                           monohydroxyphenylactic acid
25998                                   

In [ ]:
ChEBI

In [36]:
lactic;

In [38]:
# Remove l- or d- (like d-lactate) or alpha-molecule
isomer = '^\d*[dlr\+αβγδs]?-'
regular_expression = r'[()\[\],\.\{\} ]'
regular_expression_full = r'[()\+\-\[\],\.\{\} ]'

In [39]:
lactic = lactic.str.lower().str.replace(regular_expression,'', regex=True)
lactic.str.replace(isomer,'', regex=True)
lactic

422                                         s-lacticacid
16003                      r-3-4-hydroxyphenyllacticacid
16122              3-4-hydroxy-35-diiodophenyllacticacid
16373                        s-3-imidazol-5-yllacticacid
16444                                 2-acetyllacticacid
16712                                s-3-sulfolacticacid
17385                        3-4-hydroxyphenyllacticacid
17807                     3-34-dihydroxyphenyllacticacid
17943                              s-2-o-sulfolacticacid
23789                           dihydroxyphenylacticacid
24735                            hydroxyphenyllacticacid
24813                             3-indol-3-yllacticacid
24827                                   indolelacticacid
24999                                        lacticacids
25190                                 mercaptolacticacid
25396                         monohydroxyphenylacticacid
25998                                 3-phenyllacticacid
25999                          

In [35]:
lactic.str.replace(isomer,'', regex=True)


422                                         lacticacid
16003                      3-4-hydroxyphenyllacticacid
16122              4-hydroxy-35-diiodophenyllacticacid
16373                        3-imidazol-5-yllacticacid
16444                                 acetyllacticacid
16712                                3-sulfolacticacid
17385                        4-hydroxyphenyllacticacid
17807                     34-dihydroxyphenyllacticacid
17943                              2-o-sulfolacticacid
23789                         dihydroxyphenylacticacid
24735                          hydroxyphenyllacticacid
24813                             indol-3-yllacticacid
24827                                 indolelacticacid
24999                                      lacticacids
25190                               mercaptolacticacid
25396                       monohydroxyphenylacticacid
25998                                 phenyllacticacid
25999                                 phenyllacticacid
26259     